# Assignment 1 - 
**Implement a basic neural network from scratch using NumPy to classify binary data, demonstrate and compare activation functions like Sigmoid, ReLU, and Tanh, manually implement gradient descent and backpropagation with L1 and L2 regularization, and experiment with hyperparameter tuning such as learning rate, epochs, and batch size while documenting the results.**

## 1. Import Libraries and Load Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# loading dataset
data = load_breast_cancer()
X = data.data
y = data.target.reshape(-1, 1)

print("Dataset shape:", X.shape)
print("Classes:", data.target_names)

In [ ]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# normalize karna padta hai warna loss bahut bada aata hai
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# transpose for our implementation
X_train = X_train.T
X_test = X_test.T
y_train = y_train.T
y_test = y_test.T

print("Training samples:", X_train.shape[1])
print("Test samples:", X_test.shape[1])
print("Features:", X_train.shape[0])

## 2. Activation Functions

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

def tanh_act(z):
    return np.tanh(z)

def tanh_derivative(z):
    return 1 - np.tanh(z)**2

In [ ]:
# quick comparison of activations
z_vals = np.linspace(-5, 5, 200)

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.plot(z_vals, sigmoid(z_vals), color='blue')
plt.title("Sigmoid")
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(z_vals, relu(z_vals), color='green')
plt.title("ReLU")
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(z_vals, tanh_act(z_vals), color='red')
plt.title("Tanh")
plt.grid(True)

plt.suptitle("Activation Functions Comparison")
plt.tight_layout()
plt.show()

## 3. Neural Network Class

In [ ]:
class NeuralNet:
    def __init__(self, layer_sizes, activation='relu', reg_type=None, reg_lambda=0.01, lr=0.01):
        self.layer_sizes = layer_sizes
        self.activation = activation
        self.reg_type = reg_type
        self.reg_lambda = reg_lambda
        self.lr = lr
        self.params = {}
        self.loss_history = []

        np.random.seed(0)
        L = len(layer_sizes)
        for l in range(1, L):
            self.params['W' + str(l)] = np.random.randn(layer_sizes[l], layer_sizes[l-1]) * 0.01
            self.params['b' + str(l)] = np.zeros((layer_sizes[l], 1))

    def _activate(self, z):
        if self.activation == 'sigmoid':
            return sigmoid(z)
        elif self.activation == 'relu':
            return relu(z)
        elif self.activation == 'tanh':
            return tanh_act(z)

    def _activate_deriv(self, z):
        if self.activation == 'sigmoid':
            return sigmoid_derivative(z)
        elif self.activation == 'relu':
            return relu_derivative(z)
        elif self.activation == 'tanh':
            return tanh_derivative(z)

    def forward(self, X):
        cache = {'A0': X}
        L = len(self.layer_sizes) - 1
        A = X
        for l in range(1, L):
            W = self.params['W' + str(l)]
            b = self.params['b' + str(l)]
            Z = np.dot(W, A) + b
            A = self._activate(Z)
            cache['Z' + str(l)] = Z
            cache['A' + str(l)] = A

        # last layer sigmoid for binary classification
        W = self.params['W' + str(L)]
        b = self.params['b' + str(L)]
        Z = np.dot(W, A) + b
        A = sigmoid(Z)
        cache['Z' + str(L)] = Z
        cache['A' + str(L)] = A
        return A, cache

    def compute_loss(self, y_hat, y):
        m = y.shape[1]
        loss = -np.mean(y * np.log(y_hat + 1e-8) + (1 - y) * np.log(1 - y_hat + 1e-8))
        reg_term = 0
        L = len(self.layer_sizes) - 1
        if self.reg_type == 'l2':
            for l in range(1, L+1):
                reg_term += np.sum(self.params['W' + str(l)]**2)
            reg_term = (self.reg_lambda / (2 * m)) * reg_term
        elif self.reg_type == 'l1':
            for l in range(1, L+1):
                reg_term += np.sum(np.abs(self.params['W' + str(l)]))
            reg_term = (self.reg_lambda / m) * reg_term
        return loss + reg_term

    def backward(self, y, cache):
        grads = {}
        m = y.shape[1]
        L = len(self.layer_sizes) - 1
        dA = -(y / (cache['A' + str(L)] + 1e-8)) + (1 - y) / (1 - cache['A' + str(L)] + 1e-8)
        dZ = dA * sigmoid_derivative(cache['Z' + str(L)])
        grads['dW' + str(L)] = np.dot(dZ, cache['A' + str(L-1)].T) / m
        grads['db' + str(L)] = np.sum(dZ, axis=1, keepdims=True) / m
        if self.reg_type == 'l2':
            grads['dW' + str(L)] += (self.reg_lambda / m) * self.params['W' + str(L)]
        elif self.reg_type == 'l1':
            grads['dW' + str(L)] += (self.reg_lambda / m) * np.sign(self.params['W' + str(L)])
        dA_prev = np.dot(self.params['W' + str(L)].T, dZ)
        for l in reversed(range(1, L)):
            dZ = dA_prev * self._activate_deriv(cache['Z' + str(l)])
            grads['dW' + str(l)] = np.dot(dZ, cache['A' + str(l-1)].T) / m
            grads['db' + str(l)] = np.sum(dZ, axis=1, keepdims=True) / m
            if self.reg_type == 'l2':
                grads['dW' + str(l)] += (self.reg_lambda / m) * self.params['W' + str(l)]
            elif self.reg_type == 'l1':
                grads['dW' + str(l)] += (self.reg_lambda / m) * np.sign(self.params['W' + str(l)])
            dA_prev = np.dot(self.params['W' + str(l)].T, dZ)
        return grads

    def update_params(self, grads):
        L = len(self.layer_sizes) - 1
        for l in range(1, L+1):
            self.params['W' + str(l)] -= self.lr * grads['dW' + str(l)]
            self.params['b' + str(l)] -= self.lr * grads['db' + str(l)]

    def train(self, X, y, epochs=1000, batch_size=32, verbose=True):
        m = X.shape[1]
        for epoch in range(epochs):
            indices = np.random.permutation(m)
            X_shuffled = X[:, indices]
            y_shuffled = y[:, indices]
            for i in range(0, m, batch_size):
                X_batch = X_shuffled[:, i:i+batch_size]
                y_batch = y_shuffled[:, i:i+batch_size]
                y_hat, cache = self.forward(X_batch)
                grads = self.backward(y_batch, cache)
                self.update_params(grads)
            y_hat_full, _ = self.forward(X)
            loss = self.compute_loss(y_hat_full, y)
            self.loss_history.append(loss)
            if verbose and epoch % 100 == 0:
                print(f"Epoch {epoch} | Loss: {loss:.4f}")

    def predict(self, X):
        y_hat, _ = self.forward(X)
        return (y_hat > 0.5).astype(int)

    def accuracy(self, X, y):
        preds = self.predict(X)
        return np.mean(preds == y) * 100

## 4. Experiment 1 - Comparing Activation Functions

In [ ]:
results = {}
for act in ['sigmoid', 'relu', 'tanh']:
    print(f"Training with {act}...")
    model = NeuralNet([30, 16, 8, 1], activation=act, lr=0.01)
    model.train(X_train, y_train, epochs=500, batch_size=32, verbose=False)
    acc = model.accuracy(X_test, y_test)
    results[act] = {'acc': acc, 'loss': model.loss_history}
    print(f"  -> Test Accuracy: {acc:.2f}%")

In [ ]:
plt.figure(figsize=(10, 5))
for act, res in results.items():
    plt.plot(res['loss'], label=act)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curves - Different Activations")
plt.legend()
plt.grid(True)
plt.show()

## 5. Experiment 2 - L1 vs L2 Regularization

In [ ]:
reg_results = {}
for reg in [None, 'l1', 'l2']:
    label = reg if reg else 'none'
    print(f"Training with reg={label}...")
    model = NeuralNet([30, 16, 8, 1], activation='relu', reg_type=reg, reg_lambda=0.01, lr=0.01)
    model.train(X_train, y_train, epochs=500, batch_size=32, verbose=False)
    acc = model.accuracy(X_test, y_test)
    reg_results[label] = {'acc': acc, 'loss': model.loss_history}
    print(f"  -> Test Accuracy: {acc:.2f}%")

In [ ]:
plt.figure(figsize=(10, 5))
for reg, res in reg_results.items():
    plt.plot(res['loss'], label=f"reg={reg}")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curves - L1 vs L2 Regularization")
plt.legend()
plt.grid(True)
plt.show()

## 6. Experiment 3 - Hyperparameter Tuning

In [ ]:
# trying different learning rates
print("Learning Rate experiment:")
for lr in [0.001, 0.01, 0.1]:
    model = NeuralNet([30, 16, 8, 1], activation='relu', lr=lr)
    model.train(X_train, y_train, epochs=300, batch_size=32, verbose=False)
    acc = model.accuracy(X_test, y_test)
    print(f"  lr={lr} -> Accuracy: {acc:.2f}%")

In [ ]:
# trying different batch sizes
print("Batch Size experiment:")
for bs in [16, 32, 64]:
    model = NeuralNet([30, 16, 8, 1], activation='relu', lr=0.01)
    model.train(X_train, y_train, epochs=300, batch_size=bs, verbose=False)
    acc = model.accuracy(X_test, y_test)
    print(f"  batch_size={bs} -> Accuracy: {acc:.2f}%")

In [ ]:
# trying different epochs
print("Epoch experiment:")
for ep in [100, 300, 500]:
    model = NeuralNet([30, 16, 8, 1], activation='relu', lr=0.01)
    model.train(X_train, y_train, epochs=ep, batch_size=32, verbose=False)
    acc = model.accuracy(X_test, y_test)
    print(f"  epochs={ep} -> Accuracy: {acc:.2f}%")

## 7. Results Summary

In [ ]:
print("=== Final Results ===")
print("\nActivation Functions:")
for act, res in results.items():
    print(f"  {act}: {res['acc']:.2f}%")

print("\nRegularization:")
for reg, res in reg_results.items():
    print(f"  {reg}: {res['acc']:.2f}%")